# Table: Bound slope analyses — all blocks, OL and MX\n\nExtends the Panel C and Panel D analyses from Figure 3 to all 6 blocks for both\nthe OL (open-loop / blocked SNR) and MX (mixed SNR) datasets.\n\nTwo analyses, each computed per participant × block:\n\n- **Bound ~ DT** (Figure 3 Panel C): linear regression of |bound| on DT (DT ≥ 2).\n  Tests whether bounds systematically grow with decision time.\n  Null: median slope = 0.\n- **Δbound ~ bound** (Figure 3 Panel D): linear regression of Δ|bound|_t = |bound|_t − |bound|_{t−1}\n  on current |bound|_t. Tests whether large bounds tend to shrink toward the mean\n  (slope = −1 → pure regression to mean; slope = 0 → no systematic change).\n  Null: median slope = −1.\n\n**Table summary per block**: median [IQR] of per-participant slopes across participants,\nplus Wilcoxon signed-rank p-value against the stated null for each regression.\n\n**Data**: Both datasets loaded with `combine_snr=False` to retain all 6 raw\nblock/SNR combinations. Same basic quality filter as Figure 3 (`bound` finite,\n`block_index` match) plus DT ≥ 2 for the bound ~ DT regression.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import scipy.stats

from pigeon.data import get_data_table

In [2]:
data_table_ol = get_data_table(task_type='OL', combine_snr=False)
data_table_mx = get_data_table(task_type='MX', combine_snr=False)

BLOCKS = sorted(data_table_ol['block_index'].dropna().unique().astype(int))
print('OL blocks:', BLOCKS)
print('MX blocks:', sorted(data_table_mx['block_index'].dropna().unique().astype(int)))

  1: /Users/jigold/GoldWorks/Mirror_jigold/Manuscripts/2023_Pigeon/Data/Pigeon_OL/prolificcsvs/PigeonTask_2022-11-21_14h39.24.096.csv
  2: /Users/jigold/GoldWorks/Mirror_jigold/Manuscripts/2023_Pigeon/Data/Pigeon_OL/prolificcsvs/PigeonTask_2022-11-21_15h37.48.260.csv
  3: /Users/jigold/GoldWorks/Mirror_jigold/Manuscripts/2023_Pigeon/Data/Pigeon_OL/prolificcsvs/PigeonTask_2022-11-21_16h34.46.761.csv
  4: /Users/jigold/GoldWorks/Mirror_jigold/Manuscripts/2023_Pigeon/Data/Pigeon_OL/prolificcsvs/PigeonTask_2022-11-21_16h35.25.285.csv
  5: /Users/jigold/GoldWorks/Mirror_jigold/Manuscripts/2023_Pigeon/Data/Pigeon_OL/prolificcsvs/PigeonTask_2022-11-21_16h36.33.669.csv
  6: /Users/jigold/GoldWorks/Mirror_jigold/Manuscripts/2023_Pigeon/Data/Pigeon_OL/prolificcsvs/PigeonTask_2022-11-21_16h36.41.364.csv
  7: /Users/jigold/GoldWorks/Mirror_jigold/Manuscripts/2023_Pigeon/Data/Pigeon_OL/prolificcsvs/PigeonTask_2022-11-21_16h36.49.086.csv
  8: /Users/jigold/GoldWorks/Mirror_jigold/Manuscripts/2023_Pi

In [3]:
def compute_slope_stats(data_table, blocks):
    """
    For each block compute per-subject slopes for two regressions:
      bound_dt   : |bound| ~ DT  (DT >= 2, matching Figure 3 Panel C)
      delta_bound: delta|bound| ~ |bound|  (consecutive trial pairs, Panel D)

    Returns a list of row-dicts (one per block) with median, IQR, Wilcoxon p,
    and participant count for each regression.
    """
    subjects = np.sort(data_table['subject_index'].dropna().unique())
    rows = []

    for block in blocks:
        lg_block = (
            data_table['bound'].notna() &
            (data_table['block_index'] == block)
        )

        slopes_c = []  # |bound| vs DT
        slopes_d = []  # delta|bound| vs |bound|

        for subj in subjects:
            ls = lg_block & (data_table['subject_index'] == subj)

            # ── Panel C: |bound| ~ DT (DT >= 2) ──────────────────────────
            sub_c = data_table[ls & (data_table['DT'] >= 2)]
            if len(sub_c) >= 5:
                dts = sub_c['DT'].to_numpy(dtype=float)
                ab  = np.abs(sub_c['bound'].to_numpy(dtype=float))
                fin = np.isfinite(dts) & np.isfinite(ab)
                if fin.sum() >= 3:
                    try:
                        s, *_ = scipy.stats.linregress(dts[fin], ab[fin])
                        slopes_c.append(s)
                    except ValueError:
                        pass

            # ── Panel D: delta|bound| ~ |bound| ───────────────────────────
            sub_d = data_table[ls].sort_values('trial_number')
            if len(sub_d) >= 5:
                ab    = np.abs(sub_d['bound'].to_numpy(dtype=float))
                curr  = ab[1:]
                delta = curr - ab[:-1]
                fin   = np.isfinite(curr) & np.isfinite(delta)
                if fin.sum() >= 3:
                    try:
                        s, *_ = scipy.stats.linregress(curr[fin], delta[fin])
                        slopes_d.append(s)
                    except ValueError:
                        pass

        def summarize(slopes, null=0.0):
            arr = np.asarray(slopes, dtype=float)
            n   = int(np.isfinite(arr).sum())
            if n < 3:
                return dict(median=np.nan, q25=np.nan, q75=np.nan, p=np.nan, n=n)
            q25, med, q75 = np.percentile(arr, [25, 50, 75])
            try:
                _, p = scipy.stats.wilcoxon(arr - null)
            except ValueError:
                p = np.nan
            return dict(median=med, q25=q25, q75=q75, p=p, n=n)

        sc = summarize(slopes_c, null=0.0)
        sd = summarize(slopes_d, null=-1.0)
        rows.append(dict(
            block=block,
            c_n=sc['n'],
            c_median=sc['median'], c_q25=sc['q25'], c_q75=sc['q75'], c_p=sc['p'],
            d_n=sd['n'],
            d_median=sd['median'], d_q25=sd['q25'], d_q75=sd['q75'], d_p=sd['p'],
        ))

    return rows

In [4]:
rows_ol = compute_slope_stats(data_table_ol, BLOCKS)
rows_mx = compute_slope_stats(data_table_mx, BLOCKS)

In [5]:
def build_table(rows, dataset):
    def fmt_slope(med, q25, q75, n):
        if np.isnan(med):
            return f'NA  (n={n})'
        return f'{med:.4f} [{q25:.4f}, {q75:.4f}]  (n={n})'

    def fmt_p(p):
        if np.isnan(p):
            return 'NA'
        return f'{p:.4f}' if p >= 0.0001 else '<0.0001'

    records = []
    for r in rows:
        records.append({
            'Dataset': dataset,
            'Block':   r['block'],
            '|bound|~DT  median [IQR]':    fmt_slope(r['c_median'], r['c_q25'], r['c_q75'], r['c_n']),
            '|bound|~DT  p (vs 0)':        fmt_p(r['c_p']),
            'Δbound~bound  median [IQR]':   fmt_slope(r['d_median'], r['d_q25'], r['d_q75'], r['d_n']),
            'Δbound~bound  p (vs -1)':      fmt_p(r['d_p']),
        })
    return pd.DataFrame(records)


df_ol  = build_table(rows_ol, 'OL')
df_mx  = build_table(rows_mx, 'MX')
df_all = pd.concat([df_ol, df_mx], ignore_index=True)

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)
display(df_all)

,Dataset,Block,|bound|~DT median [IQR],|bound|~DT p (vs 0),Δbound~bound median [IQR],Δbound~bound p (vs -1)
0,OL,1,"0.0044 [-0.0023, 0.0172] (n=60)",0.0035,"0.9235 [0.8556, 1.0037] (n=60)",<0.0001
1,OL,2,"0.0006 [-0.0068, 0.0067] (n=59)",0.7171,"0.8726 [0.6877, 0.9605] (n=60)",<0.0001
2,OL,3,"-0.0036 [-0.0104, 0.0051] (n=56)",0.1313,"0.8731 [0.7706, 0.9755] (n=60)",<0.0001
3,OL,4,"0.0507 [0.0092, 0.0691] (n=53)",<0.0001,"0.8959 [0.7717, 1.0109] (n=60)",<0.0001
4,OL,5,"0.0130 [-0.0096, 0.0284] (n=58)",0.0084,"0.9636 [0.8870, 1.0164] (n=60)",<0.0001
5,OL,6,"0.0160 [-0.0138, 0.0354] (n=49)",0.0544,"0.9889 [0.8839, 1.0829] (n=59)",<0.0001
6,MX,1,"0.0123 [0.0018, 0.0293] (n=60)",<0.0001,"0.9493 [0.8639, 1.0140] (n=60)",<0.0001
7,MX,2,"0.0047 [-0.0034, 0.0143] (n=59)",0.0036,"0.9703 [0.8373, 1.0387] (n=60)",<0.0001
8,MX,3,"0.0002 [-0.0092, 0.0164] (n=55)",0.4609,"0.9701 [0.8776, 1.0726] (n=60)",<0.0001
9,MX,4,"0.0110 [-0.0016, 0.0240] (n=53)",0.0006,"0.9437 [0.8199, 1.0246] (n=60)",<0.0001
